# CS2 (국내반) --- Lecture 12: Graph Shooter (NumPy, Matplotlib)
### Problem Set (in-class)

*이 노트북은 채점되지 않습니다. 수업 시간에 바로 이어서 풀면 됩니다.*

- 함수 그래프를 쏴서 장애물(회색 원)을 피해 과녁(빨간 원)을 맞히는 게임
- 그래프는 발사대(검은 네모) 높이에서 출발. 장애물에 닿으면 **BLOCKED**, 화면 밖(|y| > 10)이면 **OUT**, 과녁에 닿으면 **HIT**, 과녁 위치까지 가도 안 닿으면 **MISS**
- 위에서부터 **순서대로** 실행: 앞에서 만든 함수(무기, 충돌 판정, 엔진)를 뒤 문제에서 그대로 씀
- 빈칸 `___`을 채워서 완성 (3번은 실행만). 1~5번 필수, 6번 Optional

### Setup

아래 **게임 준비** 셀을 **먼저 한 번 실행**하세요. 게임의 레벨, 랜덤 맵, 그리기 함수가 준비됩니다. (무기와 엔진은 아래 문제에서 다룹니다.)

In [ ]:
#@title 게임 준비 (먼저 한 번 실행)
# L13_game.py : levels, random maps and drawing for Graph Shooter.
# The functions YOU write (relu, insideCircle, shoot, ...) are not here.
import numpy as np
import matplotlib.pyplot as plt

np.seterr(all="ignore")       # no warnings for huge values such as np.exp(100)

# cannon: height of the cannon at x = -10
# walls:  obstacles (cx, cy, r)
# target: (tx, ty, r)
LEVELS = {
    1: dict(name="Straight Shot", cannon=0, walls=[], target=(8, 6, 0.7)),
    2: dict(name="Over the Hill", cannon=0, walls=[(0, -2, 4)], target=(8, 0, 0.7)),
    3: dict(name="Zigzag", cannon=0, walls=[(-4, 3, 2.5), (3, -3, 2.5)], target=(9, 2, 0.7)),
    4: dict(name="The Wall", cannon=-6,
            walls=[(0, -8, 2.5), (0, -4, 2), (0, 0, 2), (0, 4, 2)], target=(6, -6, 0.7)),
    5: dict(name="Step Up", cannon=-6,
            walls=[(-5, 1, 3.5), (5, -2, 3.5)], target=(8, 5, 0.7)),
    # bonus levels for the new weapons of P06
    6: dict(name="Valley", cannon=5, walls=[(-4, 8, 3.5), (4, 8, 3.5), (0, -6, 2.5)], target=(8, 4, 0.7)),
    7: dict(name="Elevator", cannon=-6, walls=[(-3, 0, 3.5), (5, 0, 3.5)], target=(8, 6, 0.7)),
    8: dict(name="Ramp", cannon=-7, walls=[(-4, -3, 2.5), (5, -6, 2.5)], target=(8, 1, 0.7)),
}

def flightPath(f, level):
    """Where the bullet really flies (for drawing): it stops at the first wall,
    when it leaves the screen, or when it reaches the target."""
    tx, ty, tr = level["target"]
    x = np.linspace(-10, tx, 2001)
    y = f(x) + 0 * x
    y = y - y[0] + level["cannon"]
    stop = np.abs(y) > 10
    for cx, cy, r in level["walls"] + [level["target"]]:
        stop = stop | (np.sqrt((x - cx) ** 2 + (y - cy) ** 2) < r)
    end = np.argmax(stop) + 1 if np.any(stop) else len(x)
    return x[:end], y[:end]

def drawLevel(level, ax=None):
    if ax is None:
        plt.figure(figsize=(6, 6))
        ax = plt.gca()
    for cx, cy, r in level["walls"]:
        ax.add_patch(plt.Circle((cx, cy), r, color="gray"))
    tx, ty, tr = level["target"]
    ax.add_patch(plt.Circle((tx, ty), tr, color="red"))
    ax.plot(-10, level["cannon"], "ks", markersize=12)
    ax.set_xlim(-10, 10)
    ax.set_ylim(-10, 10)
    ax.set_aspect("equal")
    ax.grid(True)
    ax.set_title(level["name"])

def drawShot(f, level, result):
    """Draw one level, the shot of f, and the result: drawShot(f, level, shoot(f, level))"""
    drawLevel(level)
    xs, ys = flightPath(f, level)
    plt.plot(xs, ys, "b", linewidth=2)
    plt.title(level["name"] + ": " + result)
    plt.show()

def drawLevels(fs, results):
    """fs[i]: your function for level i + 1, results[i]: its result. Draws every level."""
    plt.figure(figsize=(12, 8))
    for i in range(len(fs)):
        ax = plt.subplot(2, 3, i + 1)
        drawLevel(LEVELS[i + 1], ax)
        xs, ys = flightPath(fs[i], LEVELS[i + 1])
        ax.plot(xs, ys, "b", linewidth=2)
        ax.set_title("Level " + str(i + 1) + ": " + results[i])
    plt.tight_layout()
    plt.show()

# ---------------- random maps ----------------
def _canHit(level):
    """True if some simple shot (line, hill or step) hits the target. Used to skip impossible maps."""
    tx, ty, tr = level["target"]
    c = level["cannon"]
    tries = []
    for a in np.linspace(-1.5, 1.5, 61):
        tries.append(lambda x, a=a: a * (x + 10))
    for h in np.linspace(-14, 14, 57):
        tries.append(lambda x, h=h: (ty - c) * (x + 10) / (tx + 10) - h * (x + 10) * (x - tx) / 100)
        tries.append(lambda x, h=h: h / (1 + np.exp(-1.5 * x)))
    for f in tries:
        xs, ys = flightPath(f, level)
        if np.sqrt((xs[-1] - tx) ** 2 + (ys[-1] - ty) ** 2) < tr:
            return True
    return False

def randomLevel(seed):
    """A random map that can be cleared. The same seed always gives the same map."""
    rng = np.random.default_rng(seed)
    while True:
        walls = []
        for i in range(4):
            walls.append((rng.uniform(-6, 5), rng.uniform(-8, 8), rng.uniform(1, 2.5)))
        cannon = rng.uniform(-6, 6)
        target = (rng.uniform(6, 9), rng.uniform(-7, 7), 0.7)
        level = dict(name="Seed " + str(seed), cannon=cannon, walls=walls, target=target)
        free = True
        for cx, cy, r in walls:
            if np.sqrt((target[0] - cx) ** 2 + (target[1] - cy) ** 2) < r + 1.5:
                free = False
            if np.sqrt((-10 - cx) ** 2 + (cannon - cy) ** 2) < r + 1:
                free = False
        xs, ys = flightPath(lambda x: 0 * x, level)          # a flat shot must not win
        flatHits = np.sqrt((xs[-1] - target[0]) ** 2 + (ys[-1] - target[1]) ** 2) < target[2]
        if free and not flatHits and _canHit(level):
            return level

def drawCatalog(functions):
    """functions: a dictionary {name: function}. Draws each one."""
    x = np.linspace(-4, 4, 801)
    plt.figure(figsize=(12, 6))
    k = 0
    for name in functions:
        k = k + 1
        plt.subplot(2, 3, k)
        plt.axhline(0, color="0.7")
        plt.axvline(0, color="0.7")
        plt.plot(x, functions[name](x), linewidth=2)
        plt.xlim(-4, 4)
        plt.ylim(-3, 4)
        plt.title(name)
    plt.tight_layout()
    plt.show()

### Problem 1

(Weapons from Deep Learning)

1. Write functions `relu`, `leakyRelu`, `sigmoid`, `tanh`:
   - input parameter: an array `x`
   - return value: `x`의 각 원소에 대한 함수값의 array
     - ReLU: $x$와 $0$ 중 큰 값 (`np.maximum`)
     - Leaky ReLU: $x > 0$이면 $x$, 아니면 $0.1x$ (`np.where`)
     - sigmoid: $\dfrac{1}{1 + e^{-x}}$
     - tanh: $\dfrac{e^{x} - e^{-x}}{e^{x} + e^{-x}}$ (`np.tanh`를 쓰지 말고 공식으로)
2. Write a function `drawFunctions`:
   - action: $-5 \le x \le 5$에서 네 함수의 그래프를 그림. 보이는 범위는 x: $-5 \sim 5$, y: $-2 \sim 5$ (빈칸 채우기)

*$e^{x}$는 `np.exp(x)`. 여기서 만든 함수가 Problem 4, 5의 **무기**가 됩니다.*

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def relu(x):
    return ___

def leakyRelu(x):
    return ___

def sigmoid(x):
    return ___

def tanh(x):
    return ___

def drawFunctions():
    x = np.linspace(___, ___, 201)   # x from -5 to 5
    plt.plot(x, relu(x), label="ReLU")
    plt.plot(x, leakyRelu(x), "--", label="Leaky ReLU")
    plt.plot(x, sigmoid(x), label="sigmoid")
    plt.plot(x, tanh(x), label="tanh")
    plt.xlim(___, ___)                # show x: -5 ~ 5
    plt.ylim(___, ___)                # show y: -2 ~ 5
    plt.grid(True)
    plt.legend()
    plt.show()

p = np.array([-2, 0, 3])
print(relu(p))                  # [0 0 3]
print(leakyRelu(p))             # [-0.2  0.   3. ]
print(np.round(sigmoid(p), 3))  # [0.119 0.5   0.953]
print(np.round(tanh(p), 3))     # [-0.964  0.     0.995]
drawFunctions()

### Problem 2

(Collision)

1. Write a function `insideCircle`:
   - input parameters: arrays `x`, `y`, numbers `cx`, `cy`, `r`
   - return value: 점 `(x[i], y[i])`가 중심 `(cx, cy)`, 반지름 `r`인 원 안에 있으면 True인 mask
2. Write a function `countInside`:
   - input parameters: arrays `x`, `y`, numbers `cx`, `cy`, `r`
   - return value: 원 안에 있는 점의 개수

*`np.sum(mask)`는 True의 개수. 테스트 코드의 그림: 원 안의 점(`inside`가 True)은 빨간색, 나머지는 파란색 (그리기 코드는 주어짐).*

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def insideCircle(x, y, cx, cy, r):
    return ___

def countInside(x, y, cx, cy, r):
    return ___

x = np.array([0, 1, 2, 3, 4])
y = np.array([0, 0, 0, 0, 0])
print(insideCircle(x, y, 3, 0, 1.5))         # [False False  True  True  True]
print(np.any(insideCircle(x, y, 10, 0, 1)))  # False
print(countInside(x, y, 3, 0, 1.5))          # 3

# 500 random points and a circle (center (1, 1), radius 3)
rng = np.random.default_rng(13)
px = rng.uniform(-5, 5, 500)
py = rng.uniform(-5, 5, 500)
print(countInside(px, py, 1, 1, 3))          # 142

inside = insideCircle(px, py, 1, 1, 3)     # True: in the circle
plt.plot(px[inside], py[inside], "r.")     # in the circle: red
plt.plot(px[inside == False], py[inside == False], "b.")   # others: blue
plt.gca().set_aspect("equal")
plt.show()

### Problem 3

(The Shooting Engine) 게임 엔진은 **완성되어 있음** (빈칸 없음). 코드를 읽고 실행해서 결과 네 가지를 확인.
- `startAtCannon(y, cannon)`: 그래프를 위아래로 옮겨서 발사대 높이에서 출발
- `offScreen(y)`: |y| > 10인 값이 하나라도 있으면 True
- `touches(x, y, circle)`: 총알의 점 중 하나라도 원 안에 있으면 True (Problem 2의 `insideCircle` 이용)
- `shoot(f, level)`: 화면 밖이면 `"OUT"`, 장애물에 닿으면 `"BLOCKED"`, 과녁에 닿으면 `"HIT"`, 그 외 `"MISS"`

*테스트의 `flat`, `hill`, `down`, `steep`이 각각 어떤 결과가 나오는지 그림과 함께 확인.*

In [ ]:
import numpy as np

def startAtCannon(y, cannon):    # move the graph: y[0] becomes cannon
    return y - y[0] + cannon

def offScreen(y):                # did it leave the screen (|y| > 10)?
    return np.any(np.abs(y) > 10)

def touches(x, y, circle):       # did the bullet touch this circle?
    cx, cy, r = circle
    return np.any(insideCircle(x, y, cx, cy, r))

# the engine: read it like a sentence
def shoot(f, level):
    tx, ty, tr = level["target"]
    x = np.linspace(-10, tx, 2001)       # from x = -10 to the target
    y = startAtCannon(f(x), level["cannon"])
    if offScreen(y):
        return "OUT"
    for wall in level["walls"]:
        if touches(x, y, wall):
            return "BLOCKED"
    if touches(x, y, level["target"]):
        return "HIT"
    return "MISS"

xs = np.array([0, 1, 2])
ys = np.array([-4, 1, 6])
print(startAtCannon(ys, 3))             # [ 3  8 13]
print(offScreen(np.array([1, 12, 3])))  # True
print(touches(xs, ys, (2, 6, 0.5)))     # True

def flat(x):
    return 0 * x

def hill(x):
    return -(x + 10) * (x - 8) / 20

def down(x):
    return -0.3 * x

def steep(x):
    return 0.6 * x

level = dict(name="Test", cannon=0, walls=[(0, 0, 2)], target=(8, 0, 0.7))
print(shoot(flat, level))               # BLOCKED
print(shoot(hill, level))               # HIT
print(shoot(down, level))               # MISS
print(shoot(steep, level))              # OUT
drawShot(hill, level, shoot(hill, level))

### Problem 4

(Mission: Clear the Levels) 레벨 1~5를 **한 판씩** 깬다. 레벨마다 함수 `f1` ~ `f5`를 고쳐서 과녁을 맞히기. 과녁에 맞기만 하면 어떤 함수든 정답.
- 처음에는 `return 0 * x`: 그대로 실행해도 됨. 결과와 그림을 보고 고쳐 가면서 깨기
- 무기: Problem 1에서 만든 `relu`, `leakyRelu`, `sigmoid`, `tanh`와 `np.sin`, `np.cos`, `np.abs`, ... 도우미 함수를 `def`로 더 만들어도 됨
- 엔진: Problem 3에서 만든 `shoot`. 출력이 `HIT`이면 그 레벨 클리어

| 식 | 모양 |
|---|---|
| `a * x` | 직선, `a`가 클수록 가파름 |
| `-(x + 10) * (x - 8)` | x = -10, x = 8을 지나는 산 모양 |
| `a * np.sin(b * x)` | 물결: `a`는 높이, `b`가 클수록 촘촘 |
| `a * relu(x - c)` | x = c까지 평평, 그다음 직선 |
| `a * sigmoid(b * (x - c))` | x = c 근처에서 계단 오르기 |
| `a * tanh(x - c)` | x = c 근처에서 -a에서 a로 |
| `f(x - c)` | 오른쪽으로 c만큼 이동 |

In [ ]:
import numpy as np

# Weapons: relu, leakyRelu, sigmoid, tanh (P01), np.sin, np.cos, np.abs, ...
# Any function that hits the target is a correct answer!

#### Level 1: Straight Shot

In [ ]:
def f1(x):
    return 0 * x                     # change this!

result = shoot(f1, LEVELS[1])
print(result)                        # goal: HIT
drawShot(f1, LEVELS[1], result)

#### Level 2: Over the Hill

In [ ]:
def f2(x):
    return 0 * x                     # change this!

result = shoot(f2, LEVELS[2])
print(result)                        # goal: HIT
drawShot(f2, LEVELS[2], result)

#### Level 3: Zigzag

In [ ]:
def f3(x):
    return 0 * x                     # change this!

result = shoot(f3, LEVELS[3])
print(result)                        # goal: HIT
drawShot(f3, LEVELS[3], result)

#### Level 4: The Wall

In [ ]:
def f4(x):
    return 0 * x                     # change this!

result = shoot(f4, LEVELS[4])
print(result)                        # goal: HIT
drawShot(f4, LEVELS[4], result)

#### Level 5: Step Up

In [ ]:
def f5(x):
    return 0 * x                     # change this!

result = shoot(f5, LEVELS[5])
print(result)                        # goal: HIT
drawShot(f5, LEVELS[5], result)

### Problem 5

(Random Maps) `randomLevel(SEED)`: `SEED`에 넣은 숫자로 랜덤 맵을 만든다. 같은 숫자면 항상 같은 맵이고, 깰 수 있는 맵만 나온다.
- `myShot`을 고쳐 가며 맵을 깨기 (처음에는 `return 0 * x`)
- 깨면 `SEED`를 바꿔서 다음 맵. 여러 SEED에 도전

In [ ]:
import numpy as np

SEED = 4                # any number: a new map (1, 2, 3, ..., 2027)
level = randomLevel(SEED)

def myShot(x):
    return 0 * x                     # change this!

result = shoot(myShot, level)
print(result)                        # goal: HIT
drawShot(myShot, level, result)

### Optional Problem (Problem 6)

필수 문제와 달리 제출/검사 대상은 아니지만, 시간이 남으면 시도해보는 것을 권합니다.

(More Weapons and Bonus Levels)

1. Write functions `step`, `elu`, `softplus`, `swish`, `myAbs`:
   - step: $x > 0$이면 $1$, 아니면 $0$
   - ELU: $x > 0$이면 $x$, 아니면 $e^{x} - 1$
   - softplus: $\log(1 + e^{x})$ (`np.log`)
   - swish: $x \times \text{sigmoid}(x)$ (Problem 1의 `sigmoid` 이용)
   - myAbs: $|x|$ (`np.abs`를 쓰지 말고 `np.where`로)
2. 새 무기로 보너스 레벨 6~8을 깨기: `f6`, `f7`, `f8` (처음에는 `return 0 * x`)
   - Level 6 Valley: V자로 내려갔다 올라가기
   - Level 7 Elevator: 수직으로 점프
   - Level 8 Ramp: 평평하다가 올라가기

In [ ]:
import numpy as np

def step(x):
    return ___

def elu(x):
    return ___

def softplus(x):
    return ___

def swish(x):
    return ___

def myAbs(x):
    return ___

p = np.array([-2, 0, 3])
print(step(p))                   # [0 0 1]
print(np.round(elu(p), 3))       # [-0.865  0.     3.   ]
print(np.round(softplus(p), 3))  # [0.127 0.693 3.049]
print(np.round(swish(p), 3))     # [-0.238  0.     2.858]
print(myAbs(p))                  # [2 0 3]
drawCatalog({"step": step, "elu": elu, "softplus": softplus,
             "swish": swish, "myAbs": myAbs, "sigmoid": sigmoid})

#### Level 6: Valley

In [ ]:
def f6(x):
    return 0 * x                     # change this!

result = shoot(f6, LEVELS[6])
print(result)                        # goal: HIT
drawShot(f6, LEVELS[6], result)

#### Level 7: Elevator

In [ ]:
def f7(x):
    return 0 * x                     # change this!

result = shoot(f7, LEVELS[7])
print(result)                        # goal: HIT
drawShot(f7, LEVELS[7], result)

#### Level 8: Ramp

In [ ]:
def f8(x):
    return 0 * x                     # change this!

result = shoot(f8, LEVELS[8])
print(result)                        # goal: HIT
drawShot(f8, LEVELS[8], result)